## Load Libraries

In [ ]:
import numpy as np
import pandas as pd 
import os
from typing import Optional, Dict, Any, Tuple, List
# import matplotlib.pyplot as plt  # 本地环境无matplotlib
# import seaborn as sns  # 本地环境无seaborn
from scipy import stats
import optuna
import warnings
import polars as pl
import time
import math
import gc
import lightgbm as lgb
from scipy.stats import spearmanr
from scipy.stats import pearsonr
import joblib
import kaggle_evaluation.default_inference_server
from sklearn.model_selection import TimeSeriesSplit
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")
warnings.filterwarnings("ignore", category=RuntimeWarning, module="lightgbm")
from IPython.display import display, Markdown
from sklearn.model_selection import TimeSeriesSplit
from scipy.stats import spearmanr
from scipy.stats import pearsonr
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from catboost import CatBoostRegressor, Pool
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train_df = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
test_df = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/test.csv')

## Functions

In [ ]:
# 导入工具模块（使用相对路径）
import sys
from pathlib import Path

# 添加toollab目录到Python路径
# Kaggle环境：/kaggle/input/toollab
# 本地环境：当前目录/toollab
if Path('/kaggle/input/toollab').exists():
    # Kaggle环境
    sys.path.insert(0, '/kaggle/input')
else:
    # 本地环境
    sys.path.insert(0, str(Path.cwd()))

from toollab import (
    FactorICAnalyzer, 
    FeaturePreprocessor, 
    FeatureEngineer,
    ModelTuner,
    describe_dataset,
    missing_duplicates_analysis,
    detect_outliers,
    timer,
    safe_spearman,
    time_series_cv_splits,
    sliding_window_cv_splits,
    true_online_cv_splits,
    analyze_factor_importance_from_model
)

In [ ]:
# ============================================================
# 【数据预处理】创建滞后特征
# ============================================================

def create_lagged_features(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    为数据集添加滞后的目标变量
    
    Args:
        dataframe: 原始数据框，包含 forward_returns, risk_free_rate, market_forward_excess_returns
    
    Returns:
        添加了3个滞后特征的数据框
    """
    df = dataframe.copy()
    
    # 要创建滞后特征的目标列
    target_columns = ['forward_returns', 'risk_free_rate', 'market_forward_excess_returns']
    
    # 添加 is_scored 标记（训练集默认 False）
    if 'is_scored' not in df.columns:
        df['is_scored'] = False
    
    # 为每个目标列创建滞后特征
    for target_col in target_columns:
        if target_col in df.columns:
            df[f'lagged_{target_col}'] = df[target_col].shift(1)
            print(f"✅ 已创建 lagged_{target_col}")
        else:
            print(f"⚠️ 警告: 列 {target_col} 不存在，跳过")
    
    return df

print("✅ create_lagged_features 函数已定义")
print("   功能: 为 forward_returns, risk_free_rate, market_forward_excess_returns 创建滞后特征")
print("   用途: 在线学习时使用 lagged_forward_returns 作为上一期的真实target")

## EDA

### CONFIG

In [ ]:
# ============================================================
# 【核心配置】特征工程和模型训练配置
# ============================================================

TRAIN_PATH = '/kaggle/input/hull-tactical-market-prediction/train.csv'
LOCAL_GATEWAY_PATH = '/kaggle/input/hull-tactical-market-prediction/'

# ============================================================
# 【在线学习配置】Online Learning Settings
# ============================================================
# 【1】滑动窗口大小（天数）
TRAIN_WINDOW_SIZE = 2000  # 使用最近2000天数据训练

# 【2】在线重训练频率
ONLINE_RETRAIN_FREQ = 1  # 每1次预测重训练一次模型

# 【3】时间衰减因子（用于样本加权）
TIME_DECAY_FACTOR = 0.995  # 最近数据权重=1.0，越老数据权重越低（2000天窗口适用）

# 【4】最小训练样本数
MIN_TRAIN_SAMPLES = 100  # 至少需要100个样本才能训练（2000天窗口适用）

# 【5】因子池大小
FACTOR_POOL_SIZE = 50  # 从98个原始因子中选50个稳定因子

# 【6】使用精简特征工程
USE_SLIM_FEATURES = True  # True=精简版(~2000特征)，False=完整版(~3500特征)

# 【7】是否使用特征工程（启用！）
USE_FEATURE_ENGINEERING = True  # True=50因子→扩展特征，False=直接使用50因子

# 【8】Validation配置（已弃用 - 使用Walk-Forward代替）
VALIDATION_SPLIT = 0.0035  # 已弃用
USE_CV = False              # 不使用交叉验证

# 【9】CV策略配置（已弃用，保留向后兼容）
USE_SLIDING_WINDOW_CV = False  # 在线学习模式不使用CV
USE_REVERSE_CV = False         
CV_STEP = 10                   
USE_CV_PRUNING = False         
CV_PRUNING_STEPS = [50, 200]   
CV_PRUNING_THRESHOLD = 0.04    
CV_TIME_DECAY = 0.995          

# 【10】因子分类配置
D_FACTOR_PREFIX = 'd'          # D类因子前缀（列名以'd'开头）
KEEP_ALL_D_FACTORS = True      # 保留全部D类因子

# 【11】因子预处理配置
USE_FACTOR_PREPROCESSING = True              # 启用因子预处理（IC翻转 + 三轮变换）
SKIP_TRANSFORM_FOR_D_FACTORS = True          # D类因子跳过Log/Rank变换（只做3-Sigma）

# 【12】Lagged特征配置
LAGGED_FEATURES = [
    'lagged_forward_returns',
    'lagged_risk_free_rate',
    'lagged_market_forward_excess_returns'
]

# 【13】Walk-Forward验证配置（模拟真实推理场景）
WALKFORWARD_SKIP_SAMPLES = 4000      # 跳过前4000个样本
WALKFORWARD_TRAIN_WINDOW = 2000      # 每个窗口用2000天训练
WALKFORWARD_VAL_WINDOW = 7           # 每个窗口验证7天
WALKFORWARD_STEP_SIZE = 7            # 窗口滑动步长（7天）
WALKFORWARD_TIME_DECAY = 0.995       # 窗口得分时间衰减因子

# ============================================================
# 【传统配置】Legacy Settings
# ============================================================
# 【注意】TOP_FEATURES_FOR_FE 已被 TOP_50_FACTORS 替代
# 在线学习模式下，使用RobustFactorSelector选出的50个稳定因子
TOP_FEATURES_FOR_FE = []  # 占位符，将被 TOP_50_FACTORS 替换

# 【2】滞后期: 创建历史值特征（精简模式）
LAG_PERIODS = [1, 3, 5, 7, 14, 20]  # 6个滞后期

# 【3】滚动窗口: 计算移动统计量（精简模式）
ROLLING_WINDOWS = [3, 6, 10, 20, 60]  # 5个窗口

# 【4】预测目标: 超额收益
TARGET = 'market_forward_excess_returns'

# 【5】要排除的列: 避免数据泄漏
COLS_TO_DROP = ['forward_returns', 'risk_free_rate', 'excess_return']

# 【6】Optuna超参数搜索配置（针对Walk-Forward验证）
TUNER_SEED = 2
N_TRIALS_LIGHTGBM = 30  # LightGBM trials
N_TRIALS_CATBOOST = 20  # CatBoost trials

# 【7】时间序列交叉验证折数（已弃用）
N_SPLITS = 1  # 不使用CV，设为1
USE_CATPOOL = False 
CAT_TASK = "CPU"

- Using date_id and TimeSeriesSplit ensures that training and validation sets respect temporal order, preventing data leakage.

- Optuna allows efficient search of the hyperparameter space using Bayesian optimization. Using Spearman correlation as the optimization metric aligns model objectives with rank-based trading signals.


In [ ]:
# ============================================================
# 【鲁棒因子选择器】Robust Factor Selector
# 注意：因子筛选已提前完成，这里只导入类供参考，不执行筛选
# ============================================================
from toollab.robust_factor_selector import RobustFactorSelector

In [ ]:
# ============================================================
# 【评分指标和辅助函数】
# ============================================================
from toollab.metrics import calculate_score_metric


## Training Pipeline

In [ ]:
# 【验证Cell】模拟真实推理过程  最新版本
# 目的：完全模拟Kaggle推理时的在线学习过程
# 流程：
#   1. 初始化：训练集最后2000天
#   2. 每个test样本：预测 → 获取真实target → 更新窗口 → 重训练
#   3. 输出每步的score，验证推理逻辑

import numpy as np
import pandas as pd
import gc
import json
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

print("\n" + "="*80)
print("【推理验证模式】模拟真实在线学习过程")
print("="*80)

# ========== 环境检测 ==========
IS_KAGGLE = os.path.exists('/kaggle/input')
print(f"运行环境: {'Kaggle' if IS_KAGGLE else '本地'}")

# ========== 路径配置（与实际推理一致）==========
if IS_KAGGLE:
    # LightGBM参数（新训练的）
    PARAMS_LGBM_PATH = "/kaggle/input/online-lgb-model/best_params_lgbm.json"
    
    # 因子数据（与训练时一致）
    TOP50_FACTORS_PATH = "/kaggle/input/v5-factor/top50_stable_factors.json"
    PREPROCESSING_RULES_PATH = "/kaggle/input/v5-factor/factor_preprocessing_rules.json"
    
    # 训练数据
    TRAIN_DATA_PATH = "/kaggle/input/hull-tactical-market-prediction/train.csv"
else:
    PARAMS_LGBM_PATH = "online-lgb-model/best_params_lgbm.json"
    TOP50_FACTORS_PATH = "top50_stable_factors.json"
    PREPROCESSING_RULES_PATH = "factor_preprocessing_rules.json"
    TRAIN_DATA_PATH = TRAIN_PATH

# ========== 加载配置 ==========
print("\n加载配置文件...")

# 加载50个稳定因子
with open(TOP50_FACTORS_PATH, 'r') as f:
    TOP_50_FACTORS = json.load(f)
print(f"✅ 加载 {len(TOP_50_FACTORS)} 个因子")

# 加载预处理规则
with open(PREPROCESSING_RULES_PATH, 'r') as f:
    PREPROCESSING_RULES = json.load(f)
print(f"✅ 加载预处理规则")

# 加载LightGBM参数
with open(PARAMS_LGBM_PATH, 'r') as f:
    lgbm_params = json.load(f)
print(f"✅ 加载LightGBM超参数")

# 直接定义lagged特征（不需要从文件加载）
LAGGED_FEATURES = [
    'lagged_forward_returns',
    'lagged_risk_free_rate',
    'lagged_market_forward_excess_returns'
]
print(f"✅ 定义 {len(LAGGED_FEATURES)} 个lagged特征")

# 基础特征
BASE_FEATURES = TOP_50_FACTORS + LAGGED_FEATURES

# ========== 使用配置cell中的变量（与训练完全一致）==========
print(f"\n使用配置cell中的变量:")
print(f"   TARGET = {TARGET}")
print(f"   TRAIN_WINDOW_SIZE = {TRAIN_WINDOW_SIZE}")
print(f"   USE_FEATURE_ENGINEERING = {USE_FEATURE_ENGINEERING}")
print(f"   USE_SLIM_FEATURES = {USE_SLIM_FEATURES}")
print(f"   LAG_PERIODS = {LAG_PERIODS}")
print(f"   ROLLING_WINDOWS = {ROLLING_WINDOWS}")

# ========== 步骤1: 准备数据 ==========
print("\n--- 步骤1: 加载并准备数据 ---")

# 加载训练数据
df_train = pd.read_csv(TRAIN_DATA_PATH)
print(f"训练数据: {df_train.shape}")

# 创建lagged特征
df_train = create_lagged_features(df_train)

# 筛选特征
feature_cols = BASE_FEATURES + [TARGET, 'date_id']
df_train = df_train[[c for c in feature_cols if c in df_train.columns]]

# 应用预处理
df_train = apply_preprocessing_rules(df_train, PREPROCESSING_RULES, protected_cols=LAGGED_FEATURES)

# ========== 步骤2: 初始化历史窗口（最后2000天）==========
print("\n--- 步骤2: 初始化2000天滑动窗口 ---")

# 为了验证，我们用训练集后面的数据模拟test数据
# 取训练集最后100天作为模拟的test数据
test_start_idx = len(df_train) - 100
simulated_test_data = df_train.iloc[test_start_idx:].copy()
print(f"模拟test数据: {len(simulated_test_data)} 天")

# 重置history_df为test开始前的2000天
history_df = df_train.iloc[test_start_idx-TRAIN_WINDOW_SIZE:test_start_idx].copy()
print(f"初始历史窗口: [{test_start_idx-TRAIN_WINDOW_SIZE}:{test_start_idx}] 共 {len(history_df)} 天")
1   
# ========== 步骤3: 获取特征列名（模拟训练时的特征工程）==========
print("\n--- 步骤3: 获取特征列名 ---")

if USE_FEATURE_ENGINEERING:
    fe = FeatureEngineer(target_col=TARGET, verbose=False)
    if USE_SLIM_FEATURES:
        temp_df = fe.create_features_slim(history_df.copy())
    else:
        temp_df = fe.create_features(
            history_df.copy(),
            feature_cols=BASE_FEATURES,
            lag_periods=LAG_PERIODS,
            rolling_windows=ROLLING_WINDOWS
        )
    FEATURE_COLUMNS = [c for c in temp_df.columns if c not in [TARGET, 'date_id']]
    print(f"特征数: {len(FEATURE_COLUMNS)}")
else:
    FEATURE_COLUMNS = BASE_FEATURES
    print(f"使用基础特征: {len(FEATURE_COLUMNS)}")

# ========== 步骤4: 模拟在线推理 ==========
print("\n" + "="*80)
print("【开始模拟在线推理】")
print(f"配置:")
print(f"  - 初始窗口: 训练集最后2000天")
print(f"  - 每次重训练: 不使用sample_weight（与训练一致）")
print(f"  - 模型: LightGBM only")
print("="*80)

# 存储结果
prediction_results = []
model = None

print("\n开始逐天预测...")
print("-" * 80)

for day_idx in range(min(50, len(simulated_test_data))):  # 只测试前50天
    
    # 获取当天的test数据
    test_day = simulated_test_data.iloc[day_idx:day_idx+1].copy()
    
    print(f"\n第 {day_idx+1} 天预测:")
    
    # ========== 4.1: 更新历史窗口（如果不是第一天）==========
    if day_idx > 0:
        # 从当天的lagged获取前一天的真实target
        if 'lagged_market_forward_excess_returns' in test_day.columns:
            # 更新前一天在history中的target
            history_df.iloc[-1, history_df.columns.get_loc(TARGET)] = test_day['lagged_market_forward_excess_returns'].values[0]
            print(f"  更新Day {day_idx}的真实target: {test_day['lagged_market_forward_excess_returns'].values[0]:.6f}")
    
    # ========== 4.2: 训练模型（使用当前2000天窗口）==========
    
    # 特征工程
    if USE_FEATURE_ENGINEERING:
        fe = FeatureEngineer(target_col=TARGET, verbose=False)
        if USE_SLIM_FEATURES:
            train_feat = fe.create_features_slim(history_df.copy())
        else:
            train_feat = fe.create_features(
                history_df.copy(),
                feature_cols=BASE_FEATURES,
                lag_periods=LAG_PERIODS,
                rolling_windows=ROLLING_WINDOWS
            )
    else:
        train_feat = history_df.copy()
    
    # 删除target缺失的行
    train_feat = train_feat.dropna(subset=[TARGET])
    
    X_train = train_feat[FEATURE_COLUMNS]
    y_train = train_feat[TARGET]
    
    print(f"  训练数据: {X_train.shape[0]} 样本 × {X_train.shape[1]} 特征")
    
    # 检查维度
    if X_train.shape[1] >= X_train.shape[0] * 0.9:
        print(f"  ⚠️ 维度灾难警告: 特征数({X_train.shape[1]})接近样本数({X_train.shape[0]})!")
    
    # 训练模型（不使用sample_weight，与训练一致）
    try:
        model = LGBMRegressor(**lgbm_params)
        model.fit(X_train, y_train)  # 不使用sample_weight
        
        # 计算训练集得分
        y_pred_train = model.predict(X_train)
        train_score = calculate_score_metric(y_pred_train, y_train.values)
        print(f"  训练集Score: {train_score:.6f}")
    except Exception as e:
        print(f"  ❌ 训练失败: {e}")
        continue
    
    # ========== 4.3: 预测当前test样本 ==========
    
    # 准备预测数据（需要包含历史以计算特征）
    pred_window_size = max(ROLLING_WINDOWS) + max(LAG_PERIODS) + 5
    pred_window = history_df.tail(min(pred_window_size, len(history_df)))
    pred_window = pd.concat([pred_window, test_day]).copy()
    
    if USE_FEATURE_ENGINEERING:
        fe = FeatureEngineer(target_col=TARGET, verbose=False)
        if USE_SLIM_FEATURES:
            pred_feat = fe.create_features_slim(pred_window)
        else:
            pred_feat = fe.create_features(
                pred_window,
                feature_cols=BASE_FEATURES,
                lag_periods=LAG_PERIODS,
                rolling_windows=ROLLING_WINDOWS
            )
    else:
        pred_feat = pred_window.copy()
    
    X_pred = pred_feat.tail(1)[FEATURE_COLUMNS]
    
    # 预测
    y_pred = model.predict(X_pred)[0]
    position = 2 if y_pred > 0.001 else (1 if y_pred > 0 else 0)
    signal = convert_ret_to_signal(y_pred)
    
    # 获取真实target（用于计算score）
    y_true = test_day[TARGET].values[0]
    
    # 计算score
    score = calculate_score_metric(np.array([y_pred]), np.array([y_true]))
    
    print(f"  预测值: {y_pred:.6f} → 仓位: {position} (Signal: {signal})")
    print(f"  真实值: {y_true:.6f}")
    print(f"  Score: {score:.6f}")
    
    if score < 0:
        print(f"  ❌ 负分数!")
    
    # ========== 4.4: 将当前test数据加入历史窗口 ==========
    
    # 准备新行（包含特征但target暂时为NaN）
    new_row = test_day.copy()
    new_row[TARGET] = np.nan  # 真实target要等下一天的lagged才知道
    
    # 更新历史窗口（保持2000天）
    history_df = pd.concat([history_df.iloc[1:], new_row], ignore_index=True)
    
    print(f"  窗口滑动: 去掉最老1天，加入新1天，保持 {len(history_df)} 天")
    
    # 保存结果
    prediction_results.append({
        'day': day_idx + 1,
        'prediction': y_pred,
        'position': position,
        'signal': signal,
        'true_value': y_true,
        'score': score,
        'train_score': train_score,
        'train_samples': X_train.shape[0],
        'n_features': X_train.shape[1]
    })

print("\n" + "="*80)

# ========== 步骤5: 汇总结果 ==========
print("\n【推理结果汇总】")
print("="*80)

results_df = pd.DataFrame(prediction_results)

if len(results_df) > 0:
    print(f"\n预测天数: {len(results_df)}")
    
    print(f"\nScore统计:")
    print(f"  平均值: {results_df['score'].mean():.6f}")
    print(f"  标准差: {results_df['score'].std():.6f}")
    print(f"  最小值: {results_df['score'].min():.6f}")
    print(f"  最大值: {results_df['score'].max():.6f}")
    print(f"  负分数: {(results_df['score'] < 0).sum()} 个 ({(results_df['score'] < 0).sum()/len(results_df)*100:.1f}%)")
    
    print(f"\n训练集Score统计:")
    print(f"  平均值: {results_df['train_score'].mean():.6f}")
    print(f"  最小值: {results_df['train_score'].min():.6f}")
    print(f"  最大值: {results_df['train_score'].max():.6f}")
    
    print(f"\n仓位分布:")
    for pos in [0, 1, 2]:
        count = (results_df['position'] == pos).sum()
        print(f"  {pos}仓: {count} 次 ({count/len(results_df)*100:.1f}%)")
    
    print(f"\nSignal分布:")
    for sig in [0, 1, 2]:
        count = (results_df['signal'] == sig).sum()
        print(f"  Signal {sig}: {count} 次 ({count/len(results_df)*100:.1f}%)")
    
    print(f"\n前10天结果:")
    display_cols = ['day', 'prediction', 'position', 'signal', 'score', 'train_score', 'train_samples', 'n_features']
    print(results_df.head(10)[display_cols])
    
    print(f"\n后10天结果:")
    print(results_df.tail(10)[display_cols])
    
    # 检查维度问题
    high_dim_days = results_df[results_df['n_features'] >= results_df['train_samples'] * 0.9]
    if len(high_dim_days) > 0:
        print(f"\n⚠️ 发现 {len(high_dim_days)} 天出现维度灾难（特征数接近样本数）")
        print(high_dim_days[['day', 'train_samples', 'n_features', 'score', 'train_score']])

print("\n" + "="*80)
print("【验证完成】")
print("="*80)
print("\n关键点:")
print("1. 初始2000天窗口来自训练集末尾（不跳过4000天）")
print("2. 每预测一天，窗口滑动一天（去掉最老的，加入新的）")
print("3. 从下一天的lagged特征获取当天的真实target")
print("4. 每次都重新训练模型（真正的在线学习）")
print("5. 不使用sample_weight（与训练Walk-Forward一致）")
print("6. 使用LightGBM only（测试模式）")
print("7. 路径配置与实际推理完全一致")
print("\n如果出现负分数或维度灾难，说明有问题需要调试")
print("="*80)

# 清理内存
del df_train, history_df, results_df
gc.collect()

In [ ]:
# ============================================================
# 【在线学习训练Pipeline - Walk-Forward验证】
# ============================================================
# 新架构工作流程:
#   1. 加载原始训练数据
#   2. 创建3个lagged特征（lagged_forward_returns等）
#   2.5. 使用预加载的50个稳定因子 + 3个lagged特征（共53个基础特征）
#   3. 应用因子预处理规则（仅对50因子，保护lagged特征）
#   4. 根据USE_FEATURE_ENGINEERING决定是否进行特征工程（基于53特征）
#   5. 使用Walk-Forward验证（模拟真实推理场景）
#   6. Optuna调参（使用 Score Metric 评分）
#   7. 保存超参数（不保存模型）
#
# 关键变化:
#   - Walk-Forward: 跳过4000样本，2000天训练窗口，7天验证窗口
#   - 每个窗口在完整2000天上训练（不使用sample_weight）
#   - 时间加权：越近的窗口权重越高
#   - 模拟推理时的真实场景
# ============================================================

import gc
import json
import optuna
from sklearn.model_selection import train_test_split

# ========== 辅助函数定义（供推理时使用）==========

def calculate_score_metric(y_pred, y_true, risk_free_rate=0.0):
    """
    竞赛官方评分函数 - 调整后的夏普比率
    
    Args:
        y_pred: 预测的超额收益（连续值）
        y_true: 真实的超额收益
        risk_free_rate: 无风险利率（默认0）
    
    Returns:
        adjusted_sharpe: 调整后的夏普比率（越高越好）
    
    Notes:
        - 仓位转换：pred <= 0 → 0仓, 0 < pred <= 0.001 → 1仓, pred > 0.001 → 2仓
        - 包含波动率惩罚（超过市场1.2倍）和收益惩罚（低于市场）
    """
    if len(y_pred) == 0 or len(y_true) == 0:
        return 0.0
    
    # 1. 仓位转换
    position = np.where(y_pred > 0.001, 2.0, np.where(y_pred > 0, 1.0, 0.0))
    
    # 2. 反推forward_returns
    forward_returns = y_true + risk_free_rate
    
    # 3. 策略收益
    strategy_returns = risk_free_rate * (1 - position) + position * forward_returns
    
    # 4. 策略超额收益
    strategy_excess_returns = strategy_returns - risk_free_rate
    strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
    
    if strategy_excess_cumulative <= 0:
        return 0.0
    
    strategy_mean_excess_return = (strategy_excess_cumulative) ** (1 / len(y_pred)) - 1
    
    # 5. 策略标准差
    strategy_std = strategy_returns.std()
    
    if strategy_std == 0 or strategy_std < 1e-8:
        return 0.0
    
    # 6. 年化夏普比率
    trading_days_per_yr = 252
    sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(trading_days_per_yr)
    
    # 7. 策略年化波动率
    strategy_volatility = float(strategy_std * np.sqrt(trading_days_per_yr) * 100)
    
    # 8. 市场基准指标
    market_excess_returns = forward_returns - risk_free_rate
    market_excess_cumulative = (1 + market_excess_returns).prod()
    
    if market_excess_cumulative <= 0:
        return sharpe
    
    market_mean_excess_return = (market_excess_cumulative) ** (1 / len(y_pred)) - 1
    market_std = forward_returns.std()
    
    if market_std == 0:
        return sharpe
    
    market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)
    
    # 9. 波动率惩罚
    excess_vol = max(0, strategy_volatility / market_volatility - 1.2) if market_volatility > 0 else 0
    vol_penalty = 1 + excess_vol
    
    # 10. 收益差距惩罚
    return_gap = max(0, (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr)
    return_penalty = 1 + (return_gap**2) / 100
    
    # 11. 调整后夏普比率
    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    
    return min(float(adjusted_sharpe), 1_000_000)


def apply_preprocessing_rules(df, rules, protected_cols=None):
    """
    应用预处理规则（IC翻转 + 三轮变换）
    
    Args:
        df: 数据框
        rules: 预处理规则字典
        protected_cols: 受保护的列（不应用变换），如lagged特征
    """
    df = df.copy()
    protected_cols = protected_cols or []
    
    flip_rules = rules.get('flip_rules', [])
    for col in flip_rules:
        if col in df.columns and col not in protected_cols:
            df[col] = df[col] * (-1)
    
    transform_rules = rules.get('transform_rules', {})
    for col, transform in transform_rules.items():
        if col not in df.columns or transform == 'none' or col in protected_cols:
            continue
        try:
            if transform == 'winsorize':
                df[col] = FeaturePreprocessor.winsorize_3sigma(df[col])
            elif transform == 'log':
                df[col] = FeaturePreprocessor.log_transform(df[col])
            elif transform == 'rank':
                df[col] = FeaturePreprocessor.rank_transform(df[col])
        except:
            continue
    
    return df


def convert_ret_to_signal(ret: float) -> int:
    """将预测收益转换为离散信号 (0, 1, 2)"""
    if ret <= 0:
        return 0
    elif ret <= 0.001:
        return 1
    else:
        return 2


# ========== 训练Pipeline开始 ==========

print("\n" + "="*80)
print("【在线学习训练Pipeline - Walk-Forward验证】")
print("="*80)

# ========== 步骤1: 加载原始数据 ==========
print("\n--- 步骤1: 加载原始训练数据 ---")

df_raw = pd.read_csv(TRAIN_PATH)
print(f"✅ 原始数据形状: {df_raw.shape}")

# ========== 步骤2: 创建lagged特征 ==========
print("\n--- 步骤2: 创建lagged特征（保留forward_returns等）---")

df_with_lagged = create_lagged_features(df_raw)
print(f"✅ 数据形状（含lagged）: {df_with_lagged.shape}")
print(f"   新增列: {[c for c in df_with_lagged.columns if c.startswith('lagged_')]}")

# ========== 步骤2.5: 筛选53个基础特征（50因子 + 3lagged）==========
print("\n--- 步骤2.5: 筛选53个基础特征（50因子 + 3lagged）---")

BASE_FEATURES = TOP_50_FACTORS + LAGGED_FEATURES
feature_cols_to_keep = BASE_FEATURES + [TARGET, 'date_id']
df_selected = df_with_lagged[[c for c in feature_cols_to_keep if c in df_with_lagged.columns]].copy()
print(f"✅ 筛选后数据形状: {df_selected.shape}")
print(f"   基础特征: 50因子 + 3lagged = {len(BASE_FEATURES)}")

# ========== 步骤2.6: 应用因子预处理规则（保护lagged特征）==========
print("\n--- 步骤2.6: 应用因子预处理规则（仅对50因子，保护lagged）---")

# 保存需要保护的列
preserve_cols = {}
if TARGET in df_selected.columns:
    preserve_cols[TARGET] = df_selected[TARGET].copy()
if 'date_id' in df_selected.columns:
    preserve_cols['date_id'] = df_selected['date_id'].copy()
for lagged_col in LAGGED_FEATURES:
    if lagged_col in df_selected.columns:
        preserve_cols[lagged_col] = df_selected[lagged_col].copy()

# 应用预处理（只对50因子）
df_selected = apply_preprocessing_rules(df_selected, PREPROCESSING_RULES, protected_cols=LAGGED_FEATURES)

flip_rules = PREPROCESSING_RULES.get('flip_rules', [])
flipped_count = sum(1 for col in flip_rules if col in TOP_50_FACTORS)
print(f"  ✅ 已翻转 {flipped_count} 个因子（保护lagged特征）")

transform_rules = PREPROCESSING_RULES.get('transform_rules', {})
transformed_count = sum(1 for col in transform_rules if col in TOP_50_FACTORS and transform_rules[col] != 'none')
print(f"  ✅ 已变换 {transformed_count} 个因子（保护lagged特征）")

# 恢复保护的列
for col, values in preserve_cols.items():
    df_selected[col] = values

print(f"  ✅ 预处理完成，lagged特征保持原样")

# ========== 步骤3: 特征工程（基于53特征）==========
print("\n--- 步骤3: 特征工程（基于53个基础特征）---")

if USE_FEATURE_ENGINEERING:
    print("✅ USE_FEATURE_ENGINEERING=True，进行特征工程")
    print(f"   输入: {len(BASE_FEATURES)} 个基础特征（50因子 + 3lagged）")
    fe = FeatureEngineer(target_col=TARGET, verbose=True)
    if USE_SLIM_FEATURES:
        df_feat = fe.create_features_slim(df_selected)
    else:
        df_feat = fe.create_features(
            df_selected, 
            feature_cols=BASE_FEATURES,
            lag_periods=LAG_PERIODS,
            rolling_windows=ROLLING_WINDOWS
        )
else:
    print("✅ USE_FEATURE_ENGINEERING=False，直接使用53个基础特征")
    df_feat = df_selected.copy()

df_feat.dropna(subset=[TARGET], inplace=True)

# 准备训练数据
FEATURES = [c for c in df_feat.columns if c not in [TARGET, 'date_id']]
X = df_feat[FEATURES]
y = df_feat[TARGET]

print(f"\n✅ 数据准备完成")
print(f"   特征数: {len(FEATURES)}, 样本数: {len(X)}")

# ========== 步骤4: Walk-Forward验证函数 ==========
print("\n--- 步骤4: 定义Walk-Forward验证函数 ---")

def evaluate_walkforward(model_class, params, X, y):
    """
    Walk-Forward验证函数（模拟真实推理场景）

    Args:
        model_class: 模型类（LGBMRegressor或CatBoostRegressor）
        params: 模型超参数
        X: 特征数据
        y: 目标数据

    Returns:
        时间加权平均得分
    """
    window_scores = []
    n_windows = 0

    # 计算窗口数量
    total_samples = len(X)
    max_val_start = total_samples - WALKFORWARD_VAL_WINDOW

    for val_start in range(
        WALKFORWARD_SKIP_SAMPLES + WALKFORWARD_TRAIN_WINDOW,
        max_val_start + 1,
        WALKFORWARD_STEP_SIZE
    ):
        # 训练窗口：2000天（完整窗口，无内部split）
        train_start = val_start - WALKFORWARD_TRAIN_WINDOW
        X_train = X.iloc[train_start:val_start]
        y_train = y.iloc[train_start:val_start]

        # 验证窗口：7天
        X_val = X.iloc[val_start:val_start + WALKFORWARD_VAL_WINDOW]
        y_val = y.iloc[val_start:val_start + WALKFORWARD_VAL_WINDOW]

        # 训练模型（在完整2000天窗口上，不使用sample_weight）
        model = model_class(**params)
        model.fit(X_train, y_train)

        # 预测并评分
        y_pred = model.predict(X_val)
        score = calculate_score_metric(y_pred, y_val.values)
        window_scores.append(score)
        n_windows += 1

        # 每50个窗口输出进度
        if n_windows % 50 == 0:
            print(f"    已完成 {n_windows} 个窗口，最近得分: {score:.6f}")

    # 时间加权平均（越近的窗口权重越高）
    weights = np.array([WALKFORWARD_TIME_DECAY ** (n_windows - i - 1) for i in range(n_windows)])
    weights = weights / weights.sum()
    weighted_score = np.average(window_scores, weights=weights)

    print(f"  ✅ Walk-Forward完成: {n_windows} 个窗口, 加权得分: {weighted_score:.6f}")
    return weighted_score

expected_windows = (len(X) - WALKFORWARD_SKIP_SAMPLES - WALKFORWARD_TRAIN_WINDOW) // WALKFORWARD_STEP_SIZE
print(f"✅ Walk-Forward验证函数已定义")
print(f"   配置: 跳过{WALKFORWARD_SKIP_SAMPLES}样本, 训练窗口{WALKFORWARD_TRAIN_WINDOW}天, 验证{WALKFORWARD_VAL_WINDOW}天")
print(f"   估计窗口数: ~{expected_windows} 个")
print(f"   每个trial训练次数: {expected_windows} 次")

# ========== 步骤5: Optuna调参LightGBM（Walk-Forward验证）==========
print("\n--- 步骤5: Optuna超参数搜索 (LightGBM, Walk-Forward) ---")

def objective_lgbm(trial):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "n_estimators": trial.suggest_int("n_estimators", 400, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.08, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 13),
        "num_leaves": trial.suggest_int("num_leaves", 20, 1000),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 5.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 5.0, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": 1,
        "min_child_samples": trial.suggest_int("min_child_samples", 50, 200),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.001, 0.1, log=True),
        "random_state": TUNER_SEED,
        "verbosity": -1
    }

    # Walk-Forward验证（不使用sample_weight）
    score = evaluate_walkforward(LGBMRegressor, params, X, y)

    return score

study_lgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=TUNER_SEED))
study_lgb.optimize(objective_lgbm, n_trials=N_TRIALS_LIGHTGBM, n_jobs=1, show_progress_bar=True)

print(f"\n✅ LightGBM调参完成")
print(f"   最佳Score Metric: {study_lgb.best_value:.6f}")
print(f"   最佳参数: {list(study_lgb.best_params.items())[:5]}...")

with open('online-lgb-model/best_params_lgbm.json', 'w') as f:
    json.dump(study_lgb.best_params, f, indent=2)
print("✅ 已保存: best_params_lgbm.json")

# ========== 步骤6: Optuna调参CatBoost（Walk-Forward验证）==========
print("\n--- 步骤6: Optuna超参数搜索 (CatBoost, Walk-Forward) ---")

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 400, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.08, log=True),
        "depth": trial.suggest_int("depth", 3, 13),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.01, 5.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 0.01, 5.0, log=True),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "random_state": TUNER_SEED,
        "verbose": False,
        "task_type": CAT_TASK
    }

    # Walk-Forward验证（不使用sample_weight）
    score = evaluate_walkforward(CatBoostRegressor, params, X, y)

    return score

study_cb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=TUNER_SEED))
study_cb.optimize(objective_catboost, n_trials=N_TRIALS_CATBOOST, n_jobs=1, show_progress_bar=True)

print(f"\n✅ CatBoost调参完成")
print(f"   最佳Score Metric: {study_cb.best_value:.6f}")

with open('best_params_catboost.json', 'w') as f:
    json.dump(study_cb.best_params, f, indent=2)
print("✅ 已保存: best_params_catboost.json")

# ========== 步骤7: 保存配置 ==========
print("\n--- 步骤7: 保存特征列表和配置 ---")

with open('feature_columns.json', 'w') as f:
    json.dump(FEATURES, f, indent=2)
with open('top50_factors.json', 'w') as f:
    json.dump(TOP_50_FACTORS, f, indent=2)
with open('lagged_features.json', 'w') as f:
    json.dump(LAGGED_FEATURES, f, indent=2)
with open('factor_preprocessing_rules.json', 'w') as f:
    json.dump(PREPROCESSING_RULES, f, indent=2)

feature_engineering_config = {
    'use_feature_engineering': USE_FEATURE_ENGINEERING,
    'use_slim_features': USE_SLIM_FEATURES,
    'lag_periods': LAG_PERIODS,
    'rolling_windows': ROLLING_WINDOWS
}
with open('feature_engineering_config.json', 'w') as f:
    json.dump(feature_engineering_config, f, indent=2)

print("✅ 已保存所有配置文件")

# ========== 总结 ==========
print("\n" + "="*80)
print("【训练Pipeline完成】")
print("="*80)
print("配置:")
print(f"  验证方式: Walk-Forward (模拟真实推理场景)")
print(f"  跳过样本: {WALKFORWARD_SKIP_SAMPLES}")
print(f"  训练窗口: {WALKFORWARD_TRAIN_WINDOW} 天（完整窗口，无内部split）")
print(f"  验证窗口: {WALKFORWARD_VAL_WINDOW} 天")
print(f"  窗口步长: {WALKFORWARD_STEP_SIZE} 天")
print(f"  窗口数量: ~{expected_windows} 个")
print(f"  时间衰减: {WALKFORWARD_TIME_DECAY}")
print(f"  评分指标: Score Metric (调整后夏普比率)")
print(f"  基础特征: 53个（50因子 + 3lagged）")
print(f"  最终特征数: {len(FEATURES)}")
print(f"  样本数量: {len(X)}")
print(f"  不使用sample_weight（walk-forward本身提供时间加权）")
print("\n保存的文件:")
print("  1. best_params_lgbm.json")
print("  2. best_params_catboost.json")
print("  3. feature_columns.json")
print("  4. top50_factors.json")
print("  5. lagged_features.json")
print("  6. factor_preprocessing_rules.json")
print("  7. feature_engineering_config.json")
print("\n注意: 不保存模型，推理时用最近2000天数据重新训练")
print("="*80)

del df_raw, df_with_lagged, df_selected, df_feat, X, y
gc.collect()

## Inference

In [ ]:
# ============================================================
# 【在线学习推理Pipeline - 真正的在线学习版本（53特征）】
# ============================================================
# 核心改进：
#   - 利用 test 数据的 lagged_market_forward_excess_returns（上一期真实target）
#   - 使用53个基础特征（50因子 + 3lagged）进行训练和预测
#   - 每次预测后，将新数据（包含真实target）加入训练集
#   - 实现真正的增量学习（训练集不断更新）
#   - 不使用sample_weight（与训练保持一致）
#
# 关键变化：
#   - 初始化时创建滞后特征，保留3个lagged列
#   - 预处理时保护lagged特征不被变换
#   - predict 函数中使用 lagged_market_forward_excess_returns 更新历史
#   - 历史窗口扩展到2000天
# ============================================================

print("\n" + "="*80)
print("【在线学习推理系统初始化 - 53特征（50因子+3lagged）】")
print("="*80)

# ========== 环境检测 ==========
IS_KAGGLE = os.path.exists('/kaggle/input')
print(f"运行环境: {'Kaggle' if IS_KAGGLE else '本地'}")

# ========== 路径配置 ==========
if IS_KAGGLE:
    PARAMS_LGBM_PATH = "/kaggle/input/v5-mdoel1/best_params_lgbm.json"
    PARAMS_CAT_PATH = "/kaggle/input/v5-mdoel1/best_params_catboost.json"
    FEATURE_COLS_PATH = "/kaggle/input/v5-mdoel1/feature_columns.json"
    TOP50_FACTORS_PATH = "/kaggle/input/v5-mdoel1/top50_factors.json"
    LAGGED_FEATURES_PATH = "/kaggle/input/v5-mdoel1/lagged_features.json"
    PREPROCESSING_RULES_PATH = "/kaggle/input/v5-mdoel1/factor_preprocessing_rules.json"
    FEATURE_ENGINEERING_CONFIG_PATH = "/kaggle/input/v5-mdoel1/feature_engineering_config.json"
    TRAIN_DATA_PATH = "/kaggle/input/hull-tactical-market-prediction/train.csv"
else:
    PARAMS_LGBM_PATH = "online-lgb-model/best_params_lgbm.json"
    PARAMS_CAT_PATH = "best_params_catboost.json"
    FEATURE_COLS_PATH = "feature_columns.json"
    TOP50_FACTORS_PATH = "top50_factors.json"
    LAGGED_FEATURES_PATH = "lagged_features.json"
    PREPROCESSING_RULES_PATH = "factor_preprocessing_rules.json"
    FEATURE_ENGINEERING_CONFIG_PATH = "feature_engineering_config.json"
    TRAIN_DATA_PATH = TRAIN_PATH

# ========== 加载配置文件 ==========
print(f"\n加载配置文件...")

with open(TOP50_FACTORS_PATH, 'r') as f:
    TOP_50_FACTORS = json.load(f)
print(f"✅ 已加载 {len(TOP_50_FACTORS)} 个稳定因子")

# 加载lagged特征列表（如果文件不存在则使用默认值）
try:
    with open(LAGGED_FEATURES_PATH, 'r') as f:
        LAGGED_FEATURES = json.load(f)
    print(f"✅ 已加载 {len(LAGGED_FEATURES)} 个lagged特征")
except FileNotFoundError:
    LAGGED_FEATURES = [
        'lagged_forward_returns',
        'lagged_risk_free_rate',
        'lagged_market_forward_excess_returns'
    ]
    print(f"⚠️ 使用默认lagged特征列表（共{len(LAGGED_FEATURES)}个）")

with open(PREPROCESSING_RULES_PATH, 'r') as f:
    PREPROCESSING_RULES = json.load(f)
print(f"✅ 已加载预处理规则")

if 'lgbm_params' not in globals():
    with open(PARAMS_LGBM_PATH, 'r') as f:
        lgbm_params = json.load(f)
    print(f"✅ 已加载LightGBM超参数")

if 'catboost_params' not in globals():
    with open(PARAMS_CAT_PATH, 'r') as f:
        catboost_params = json.load(f)
    print(f"✅ 已加载CatBoost超参数")

with open(FEATURE_COLS_PATH, 'r') as f:
    FEATURE_COLUMNS = json.load(f)
print(f"✅ 已加载 {len(FEATURE_COLUMNS)} 个特征列")

with open(FEATURE_ENGINEERING_CONFIG_PATH, 'r') as f:
    feature_engineering_config = json.load(f)
    USE_FEATURE_ENGINEERING = feature_engineering_config['use_feature_engineering']
    USE_SLIM_FEATURES = feature_engineering_config['use_slim_features']
    LAG_PERIODS = feature_engineering_config['lag_periods']
    ROLLING_WINDOWS = feature_engineering_config['rolling_windows']
print(f"✅ 已加载特征工程配置")

# 定义53个基础特征
BASE_FEATURES = TOP_50_FACTORS + LAGGED_FEATURES
print(f"\n✅ 基础特征: {len(BASE_FEATURES)} 个（50因子 + 3lagged）")

# ========== 初始化历史数据（关键改进）==========
print(f"\n初始化历史数据窗口（启用在线学习，保留lagged特征）...")

df_train_full = pd.read_csv(TRAIN_DATA_PATH)

# 【关键】为训练集创建滞后特征
print("  创建滞后特征（lagged_forward_returns, lagged_risk_free_rate, lagged_market_forward_excess_returns）...")
df_train_full = create_lagged_features(df_train_full)

# 保留50因子 + 3lagged + target + date_id
keep_cols = [c for c in [TARGET, 'date_id'] + BASE_FEATURES if c in df_train_full.columns]
history_df = df_train_full[keep_cols].tail(2000).copy()

print(f"✅ 历史数据初始化完成")
print(f"   保留最近: {len(history_df)} 行")
print(f"   列数: {len(history_df.columns)}")
print(f"   包含50因子 + 3lagged特征")

del df_train_full
gc.collect()

# ========== 全局变量 ==========
cached_lgbm_model = None
cached_catboost_model = None
prediction_counter = 0
last_retrain_counter = -1

print(f"\n✅ 初始化完成！")
print(f"   在线重训练频率: 每 {ONLINE_RETRAIN_FREQ} 次预测")
print(f"   训练窗口大小: {TRAIN_WINDOW_SIZE} 天")
print(f"   基础特征数: {len(BASE_FEATURES)}（50因子 + 3lagged）")
print("="*80)

# ========== 主预测函数（真正的在线学习，53特征）==========

def predict(test_df_pl: pl.DataFrame) -> float:
    """
    真正的在线学习预测函数（使用53个基础特征）
    
    核心改进：
    1. 从 test_df 提取 50因子 + 3lagged特征
    2. 将新数据（50因子 + 3lagged + 真实target）加入历史
    3. 训练集不断更新，实现真正的增量学习
    4. 预处理时保护lagged特征不被变换
    5. 不使用sample_weight（与训练保持一致）
    """
    global history_df, cached_lgbm_model, cached_catboost_model
    global prediction_counter, last_retrain_counter
    
    # ========== 步骤1: 提取新数据（50因子 + 3lagged）==========
    test_df_pd = test_df_pl.to_pandas()
    
    # 提取53个基础特征
    test_cols = [c for c in ['date_id'] + BASE_FEATURES if c in test_df_pd.columns]
    new_data = test_df_pd[test_cols].copy()
    
    # 【关键】提取上一期的真实target（从lagged_market_forward_excess_returns）
    if 'lagged_market_forward_excess_returns' in test_df_pd.columns:
        new_data[TARGET] = test_df_pd['lagged_market_forward_excess_returns'].values
        print(f"[预测 #{prediction_counter+1}] 获取真实target: {new_data[TARGET].values[0]:.6f}")
    else:
        # 如果没有lagged值（第一次预测），使用NaN
        new_data[TARGET] = np.nan
        print(f"[预测 #{prediction_counter+1}] 无lagged_target（首次预测）")
    
    # 应用预处理规则（保护lagged特征）
    new_data = apply_preprocessing_rules(new_data, PREPROCESSING_RULES, protected_cols=LAGGED_FEATURES)
    
    # 【关键】将新数据（包含真实target）加入历史（保留最近2000天）
    history_df = pd.concat([history_df, new_data], ignore_index=True).tail(2000)
    
    prediction_counter += 1
    
    # ========== 步骤2: 判断是否需要重训练 ==========
    need_retrain = (
        cached_lgbm_model is None or 
        cached_catboost_model is None or
        (prediction_counter - last_retrain_counter) >= ONLINE_RETRAIN_FREQ
    )
    
    if need_retrain:
        print(f"  触发在线重训练（使用最近{TRAIN_WINDOW_SIZE}天数据，53个基础特征）...")
        
        # 准备训练数据（使用不断更新的历史数据）
        train_window = history_df.tail(min(TRAIN_WINDOW_SIZE, len(history_df))).copy()
        
        # 特征工程（可选，基于53个基础特征）
        if USE_FEATURE_ENGINEERING:
            fe = FeatureEngineer(target_col=TARGET, verbose=False)
            if USE_SLIM_FEATURES:
                feat_df = fe.create_features_slim(train_window)
            else:
                feat_df = fe.create_features(
                    train_window,
                    feature_cols=BASE_FEATURES,
                    lag_periods=LAG_PERIODS,
                    rolling_windows=ROLLING_WINDOWS
                )
        else:
            feat_df = train_window.copy()
        
        # 删除target缺失的行
        feat_df.dropna(subset=[TARGET], inplace=True)
        
        if len(feat_df) >= MIN_TRAIN_SAMPLES:
            X_train = feat_df[FEATURE_COLUMNS]
            y_train = feat_df[TARGET]
            
            # 不使用sample_weight（与训练Walk-Forward保持一致）
            
            print(f"  训练样本: {len(X_train)} | 特征: {X_train.shape[1]} | 最新target: {y_train.iloc[-1]:.6f}")
            
            # 重新训练
            try:
                cached_lgbm_model = LGBMRegressor(**lgbm_params)
                cached_lgbm_model.fit(X_train, y_train)  # 不使用sample_weight
                
                cached_catboost_model = CatBoostRegressor(**catboost_params, verbose=False)
                cached_catboost_model.fit(X_train, y_train)  # 不使用sample_weight
                
                print(f"  ✅ 模型重训练完成")
                last_retrain_counter = prediction_counter
            except Exception as e:
                print(f"  ❌ 训练失败: {e}")
            
            del feat_df, X_train, y_train
            gc.collect()
        else:
            print(f"  ⚠️ 样本数不足({len(feat_df)} < {MIN_TRAIN_SAMPLES})，跳过重训练")
    
    # ========== 步骤3: 预测当前样本 ==========
    # 特征工程（预测样本，基于53个基础特征）
    if USE_FEATURE_ENGINEERING:
        pred_window_size = max(ROLLING_WINDOWS) + max(LAG_PERIODS) + 5
        pred_window = history_df.tail(min(pred_window_size, len(history_df))).copy()
        
        fe = FeatureEngineer(target_col=TARGET, verbose=False)
        if USE_SLIM_FEATURES:
            pred_feat = fe.create_features_slim(pred_window)
        else:
            pred_feat = fe.create_features(
                pred_window,
                feature_cols=BASE_FEATURES,
                lag_periods=LAG_PERIODS,
                rolling_windows=ROLLING_WINDOWS
            )
    else:
        pred_feat = history_df.tail(1).copy()
    
    X_pred = pred_feat.tail(1)[FEATURE_COLUMNS]
    
    # 预测
    try:
        pred_lgbm = float(cached_lgbm_model.predict(X_pred)[0])
        pred_cat = float(cached_catboost_model.predict(X_pred)[0])
        
        blended_pred = 0.6 * pred_lgbm + 0.4 * pred_cat
        signal = convert_ret_to_signal(blended_pred)
        
        print(f"  预测: LGBM={pred_lgbm:.6f}, CAT={pred_cat:.6f}, Blend={blended_pred:.6f}, Signal={signal}\n")
    except Exception as e:
        print(f"  ❌ 预测失败: {e}")
        signal = 1
    
    gc.collect()
    return float(signal)


# ========== Kaggle推理服务器 ==========
print("\n初始化Kaggle推理服务器...")

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    print("🚀 启动竞赛推理服务...")
    inference_server.serve()
else:
    print("🧪 启动本地测试网关...")
    inference_server.run_local_gateway((LOCAL_GATEWAY_PATH,))

In [ ]:
bushi1